# Colab Minimal Pipeline (Smoke Test)

Minimal run order for this project:
1. Install dependencies
2. (Optional) Login to Weights & Biases
3. Train Stage 1 (`train_hysteresis.py`)
4. Train Stage 2 (`train_quantification.py`)
5. Train Stage 3 (`train_mixed.py`)


In [ ]:
# === EDIT THIS IF NEEDED ===
PROJECT_DIR = "/content/intership-project"

import os
assert os.path.isdir(PROJECT_DIR), f"Project dir not found: {PROJECT_DIR}"
%cd /content/intership-project
print("Project dir:", PROJECT_DIR)


In [ ]:
# Install dependencies
!pip install -q --upgrade pip
!pip install -q -r requirements.txt


In [ ]:
# Optional: Weights & Biases
USE_WANDB = True  # set False to disable

if USE_WANDB:
    import wandb
    wandb.login()
    WANDB_FLAG = "--use_wandb"
else:
    WANDB_FLAG = ""

print("WANDB_FLAG:", WANDB_FLAG)


In [ ]:
# Data quick check (must exist before training)
required_paths = [
    "Data/Hysteresis/info_w_mander_big.csv",
    "Data/Hysteresis/id_list_small.json",
    "Data/DamagePattern/label.csv",
    "Data/DamagePattern/id_list.json",
]

for p in required_paths:
    print(p, "OK" if os.path.exists(p) else "MISSING")


In [ ]:
# Stage 1: Hysteretic prediction (smoke test)
!python train_hysteresis.py --architecture HystereticGRU --epoch 3 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# Stage 2: Damage quantification (smoke test)
!python train_quantification.py --backbone vit_b_16 --epoch 3 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# Auto-find latest Stage 1 and Stage 2 run folders
import glob

def latest_run(base_glob):
    runs = sorted(glob.glob(base_glob), key=os.path.getmtime)
    assert runs, f"No runs found for: {base_glob}"
    return runs[-1]

extractor_path = latest_run("results/HystereticPrediction/HystereticGRU/*")
regressor_path = latest_run("results/DamageQuantification/vit_b_16/*")
print("extractor_path =", extractor_path)
print("regressor_path =", regressor_path)


In [ ]:
# Stage 3: Mixed training (smoke test)
!python train_mixed.py \
  --architecture HystereticCGAN \
  --loss_type SGAN \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning False \
  --direction_conditioning False \
  --add_DI_loss False \
  --image_size 128 \
  --z_dim 100 \
  --G_h_size 128 \
  --D_h_size 128 \
  --SELU False \
  --spectral_D False \
  --no_batch_norm False \
  --epoch 3 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}
